# The Translator — Combining v1 and v2

**Claim:** Both Translator versions land in the same derived 16-dim prime-channel space, so they can be cross-tested directly.

The joint that makes this work: `hypervector(t)[0:16] == channel_signature(t)` exactly. The DisCoCat verb tensor (16³) and the VSA hypervector (4096) are the same numbers reshaped.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../../..'))
from ValaQuenta.modules.translator_discocat import DisCoCatTranslator
from ValaQuenta.modules.translator_vsa import VSATranslator
from ValaQuenta.modules.translator_common.maths import (
    compare_engines, verify_harmonic_reduction)
print('both loaded')

## The shared-space joint — must hold, or the two must not be combined

In [ ]:
for t in ['dog','bites','man']:
    r = verify_harmonic_reduction(t)
    print(t, 'residual', r['max_residual'], 'matches', r['matches'])

## Cross-engine comparison

`compare_engines` reports; it does not score or tune. Kendall tau is rank agreement between the two versions — a measurement, not a target.

In [ ]:
triples = [('dog','bites','man'),('man','bites','dog'),
           ('cat','runs','water'),('water','runs','cat')]
r = compare_engines(DisCoCatTranslator(), VSATranslator(), triples)
for row in r['per_triple']:
    print(row['triple'], round(row['cos_between_engines'],6))
print('kendall tau', r['kendall_tau'])

## Standing result (2026-07-28)

Both engines' concept vectors are crowded: mean |cos| ≈ 0.96 between distinct tokens. Constituent recovery sits at chance. Phase 22 of VAPMIP's `Tuning-the-Engine.md` makes under-resolution the standing first hypothesis for a flat 16D result — **tested here and rejected**: the same encoder at 4096 dims is just as crowded as at 16. The cause is a common mode in the encoder (~85% of every token vector is a shared direction), which is dimension-independent.

Left in place per Prime Directive #2.

In [ ]:
from ValaQuenta.modules.translator_common.maths import channel_signature, hypervector, cosine
toks = ['dog','man','bites','cat','runs','water','hot','cold','love','true']
for f,lab,dim in ((channel_signature,'16-dim',16),(hypervector,'4096-dim',4096)):
    vs = {t:f(t) for t in toks}
    cs = [abs(cosine(vs[a],vs[b])) for i,a in enumerate(toks) for b in toks[i+1:]]
    print(f'{lab:9} dim={dim:5} mean|cos|={sum(cs)/len(cs):.4f}')